[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/07_joint_distributions_and_multivariate_normal/first_principles.ipynb)

# Topic 07: Joint Distributions and the Multivariate Normal

## 1. First-Principles Intuition & Motivation

Nothing interesting is one-dimensional. A pixel is meaningless without its neighbours; a stock return matters only relative to the market; a sensor reading is useful only when fused with the others. The mathematical object that carries all of this is the **joint distribution**,

$$
F_{\mathbf{X}}(\mathbf{x}) = P\left(X_1 \le x_1, \ldots, X_d \le x_d\right),
$$

and the essential fact about it is asymmetric: the joint always determines the marginals (integrate out what you do not want), but the marginals never determine the joint. The missing ingredient is the **dependence structure**, and Sklar's theorem makes the separation exact — any joint law factors into its marginals plus a copula, and the copula is a free modeling choice.

Dependence is also what makes multivariate probability hard. A general joint density over $d$ variables is an object on $\mathbb{R}^d$; estimating it from data is hopeless without structure. Two kinds of structure rescue the situation. **Conditional independence** cuts the joint into small factors (the basis of graphical models). **Gaussianity** collapses everything into a mean vector and a covariance matrix, and — remarkably — keeps every operation we care about inside the family.

### 1.1 Why the Multivariate Normal Is the Only Fully Tractable Multivariate Family

For $\mathcal{N}(\boldsymbol\mu, \Sigma)$ every standard operation has a closed form, and each one is a matrix computation:

| Operation | Result | Cost |
|---|---|---|
| Affine map $A\mathbf{X}+\mathbf{b}$ | $\mathcal{N}\left(A\boldsymbol\mu+\mathbf{b},\, A\Sigma A^{\top}\right)$ | one matrix product |
| Marginalize a block | drop the corresponding rows/columns of $\boldsymbol\mu, \Sigma$ | free |
| Condition on a block | Gaussian, linear mean shift, covariance $\Sigma_{11}-\Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$ | one solve |
| Sum of independent Gaussians | add means, add covariances | free |
| Sample | $\boldsymbol\mu + L\mathbf{z}$ with $\Sigma = LL^{\top}$ | one Cholesky |
| Conditional independence test | check whether $\left(\Sigma^{-1}\right)_{ij} = 0$ | one inverse |

No other multivariate family closes under all of these. That is why Gaussian assumptions are the default in Kalman filtering, Gaussian processes, linear-Gaussian state-space models, and the forward/reverse kernels of diffusion models — not because data is Gaussian, but because the algebra survives contact with the algorithm.

Two matrices tell the whole story. The **covariance** $\Sigma$ describes marginal co-variation: its eigenvectors are principal components and its Cholesky factor generates samples. The **precision** $\Lambda = \Sigma^{-1}$ describes conditional structure: $\Lambda_{ij} = 0$ exactly when $X_i \perp X_j$ given all other coordinates. Marginal and conditional independence are genuinely different questions, and they live in different matrices.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 2.1 (Joint CDF and Density).** For a random vector $\mathbf{X} = (X_1,\ldots,X_d)$,

$$
F_{\mathbf{X}}(\mathbf{x}) = P\left(X_1 \le x_1, \ldots, X_d \le x_d\right), \qquad f_{\mathbf{X}}(\mathbf{x}) = \frac{\partial^d F_{\mathbf{X}}}{\partial x_1 \cdots \partial x_d}
$$

when the mixed partial exists. Then $P(\mathbf{X} \in B) = \int_B f_{\mathbf{X}}(\mathbf{x})\,d\mathbf{x}$ for Borel $B \subseteq \mathbb{R}^d$.

**Definition 2.2 (Marginal and Conditional).** The marginal of a sub-vector integrates out the rest,

$$
f_{X}(x) = \int_{\mathbb{R}} f_{X,Y}(x,y)\,dy, \qquad f_{Y \mid X}(y \mid x) = \frac{f_{X,Y}(x,y)}{f_X(x)} \quad \text{wherever } f_X(x) \gt 0.
$$

For each fixed $x$, $f_{Y \mid X}(\cdot \mid x)$ is a genuine density in $y$.

**Definition 2.3 (Independence).** $X$ and $Y$ are independent iff any of the following equivalent conditions holds for all $x, y$:

$$
F_{X,Y}(x,y) = F_X(x)F_Y(y), \qquad f_{X,Y}(x,y) = f_X(x)f_Y(y), \qquad f_{Y\mid X}(y \mid x) = f_Y(y).
$$

A practical sufficient criterion: if $f_{X,Y}(x,y) = g(x)h(y)$ on a **product-shaped support**, then $X \perp Y$. The support condition is essential — $f(x,y) = 2$ on the triangle $0 \lt x \lt y \lt 1$ factors trivially as $2 \times 1$ yet the variables are dependent, because the region couples them.

**Definition 2.4 (Mean Vector and Covariance Matrix).** For a square-integrable random vector,

$$
\boldsymbol{\mu} = E[\mathbf{X}] \in \mathbb{R}^d, \qquad \Sigma = \mathrm{Cov}(\mathbf{X}) = E\left[\left(\mathbf{X}-\boldsymbol\mu\right)\left(\mathbf{X}-\boldsymbol\mu\right)^{\top}\right] = E\left[\mathbf{X}\mathbf{X}^{\top}\right] - \boldsymbol\mu\boldsymbol\mu^{\top}.
$$

**Theorem 2.5 (Properties of $\Sigma$).** $\Sigma$ is symmetric and positive semidefinite; it is singular exactly when some linear combination $\mathbf{a}^{\top}\mathbf{X}$ is almost surely constant. Under an affine map,

$$
E\left[A\mathbf{X}+\mathbf{b}\right] = A\boldsymbol\mu + \mathbf{b}, \qquad \mathrm{Cov}\left(A\mathbf{X}+\mathbf{b}\right) = A\Sigma A^{\top},
$$

and in particular $\mathrm{Var}\left(\mathbf{a}^{\top}\mathbf{X}\right) = \mathbf{a}^{\top}\Sigma\mathbf{a} \ge 0$, which *is* the positive-semidefiniteness statement.

**Definition 2.6 (Multivariate Normal).** $\mathbf{X} \sim \mathcal{N}_d(\boldsymbol\mu, \Sigma)$ with $\Sigma \succ 0$ if

$$
f_{\mathbf{X}}(\mathbf{x}) = \frac{1}{(2\pi)^{d/2}\left(\det\Sigma\right)^{1/2}}\exp\left(-\frac{1}{2}\left(\mathbf{x}-\boldsymbol\mu\right)^{\top}\Sigma^{-1}\left(\mathbf{x}-\boldsymbol\mu\right)\right).
$$

The quadratic form $\Delta^2(\mathbf{x}) = (\mathbf{x}-\boldsymbol\mu)^{\top}\Sigma^{-1}(\mathbf{x}-\boldsymbol\mu)$ is the squared **Mahalanobis distance**; its level sets are the ellipsoidal contours of the density, with axes along the eigenvectors of $\Sigma$ and semi-axis lengths proportional to $\sqrt{\lambda_i}$.

**Definition 2.7 (General / Degenerate Case).** $\mathbf{X}$ is multivariate normal iff $\mathbf{a}^{\top}\mathbf{X}$ is univariate normal (possibly degenerate) for **every** $\mathbf{a} \in \mathbb{R}^d$. Equivalently $\mathbf{X} \overset{d}{=} \boldsymbol\mu + A\mathbf{Z}$ with $\mathbf{Z} \sim \mathcal{N}(\mathbf{0}, I)$; then $\Sigma = AA^{\top}$ and no density exists when $\Sigma$ is singular.

**Theorem 2.8 (The Four Gaussian Identities).** Partition $\mathbf{X} = \begin{pmatrix}\mathbf{X}_1\\ \mathbf{X}_2\end{pmatrix} \sim \mathcal{N}\left(\begin{pmatrix}\boldsymbol\mu_1\\ \boldsymbol\mu_2\end{pmatrix}, \begin{pmatrix}\Sigma_{11} & \Sigma_{12}\\ \Sigma_{21} & \Sigma_{22}\end{pmatrix}\right)$.

1. **Affine closure**: $A\mathbf{X}+\mathbf{b} \sim \mathcal{N}\left(A\boldsymbol\mu+\mathbf{b},\ A\Sigma A^{\top}\right)$ for any $A$, $\mathbf{b}$.
2. **Marginalization**: $\mathbf{X}_1 \sim \mathcal{N}\left(\boldsymbol\mu_1, \Sigma_{11}\right)$ — just read off the blocks.
3. **Conditioning**: $\mathbf{X}_1 \mid \mathbf{X}_2 = \mathbf{x}_2 \sim \mathcal{N}\left(\boldsymbol\mu_{1\mid 2}, \Sigma_{1\mid 2}\right)$ with

$$
\boldsymbol\mu_{1\mid2} = \boldsymbol\mu_1 + \Sigma_{12}\Sigma_{22}^{-1}\left(\mathbf{x}_2 - \boldsymbol\mu_2\right), \qquad \Sigma_{1\mid2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}.
$$

The conditional covariance is the **Schur complement** of $\Sigma_{22}$ and does not depend on the observed $\mathbf{x}_2$.

4. **Independence from correlation**: for a jointly Gaussian vector, $\Sigma_{12} = 0 \iff \mathbf{X}_1 \perp \mathbf{X}_2$.

**Theorem 2.9 (Precision Matrix and Conditional Independence).** Let $\Lambda = \Sigma^{-1}$. Then for $i \ne j$,

$$
\Lambda_{ij} = 0 \iff X_i \perp X_j \;\bigm|\; \mathbf{X}_{\setminus\{i,j\}},
$$

and the partial correlation is $\rho_{ij \mid \text{rest}} = -\dfrac{\Lambda_{ij}}{\sqrt{\Lambda_{ii}\Lambda_{jj}}}$.

**Theorem 2.10 (Multivariate Change of Variables).** If $\mathbf{Y} = g(\mathbf{X})$ with $g$ a diffeomorphism, then

$$
f_{\mathbf{Y}}(\mathbf{y}) = f_{\mathbf{X}}\left(g^{-1}(\mathbf{y})\right)\left\lvert \det J_{g^{-1}}(\mathbf{y}) \right\rvert.
$$

**Theorem 2.11 (Sklar).** For any joint CDF $F$ with continuous marginals $F_1,\ldots,F_d$ there is a **unique** copula $C : [0,1]^d \to [0,1]$ — itself a CDF with uniform marginals — such that $F(\mathbf{x}) = C\left(F_1(x_1),\ldots,F_d(x_d)\right)$. Conversely any copula plus any marginals defines a valid joint law.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1: Affine Closure and the Covariance Transformation Rule

**Claim.** If $\mathbf{X} \sim \mathcal{N}(\boldsymbol\mu, \Sigma)$ then $\mathbf{Y} = A\mathbf{X}+\mathbf{b} \sim \mathcal{N}\left(A\boldsymbol\mu+\mathbf{b}, A\Sigma A^{\top}\right)$.

**Proof (moments, valid always).** Linearity of expectation gives $E[\mathbf{Y}] = A\boldsymbol\mu+\mathbf{b}$, and

$$
\mathrm{Cov}(\mathbf{Y}) = E\left[A(\mathbf{X}-\boldsymbol\mu)\left(A(\mathbf{X}-\boldsymbol\mu)\right)^{\top}\right] = A\,E\left[(\mathbf{X}-\boldsymbol\mu)(\mathbf{X}-\boldsymbol\mu)^{\top}\right]A^{\top} = A\Sigma A^{\top}.
$$

Note this part uses no normality at all — it holds for any random vector with finite second moments.

**Proof (normality, via the characterization).** For any $\mathbf{a}$, $\mathbf{a}^{\top}\mathbf{Y} = \left(A^{\top}\mathbf{a}\right)^{\top}\mathbf{X} + \mathbf{a}^{\top}\mathbf{b}$, which is a univariate normal shifted by a constant, hence normal. Since every linear combination of $\mathbf{Y}$ is univariate normal, Definition 2.7 gives that $\mathbf{Y}$ is multivariate normal, and its parameters are the moments computed above. $\blacksquare$

**Proof (density, when $A$ is square and invertible).** Apply Theorem 2.10 with $g^{-1}(\mathbf{y}) = A^{-1}(\mathbf{y}-\mathbf{b})$ and $\left\lvert \det J\right\rvert = \left\lvert \det A\right\rvert^{-1}$. The exponent becomes

$$
-\frac{1}{2}\left(A^{-1}(\mathbf{y}-\mathbf{b})-\boldsymbol\mu\right)^{\top}\Sigma^{-1}\left(A^{-1}(\mathbf{y}-\boldsymbol{b})-\boldsymbol\mu\right) = -\frac{1}{2}\left(\mathbf{y}-\mathbf{b}-A\boldsymbol\mu\right)^{\top}\left(A\Sigma A^{\top}\right)^{-1}\left(\mathbf{y}-\mathbf{b}-A\boldsymbol\mu\right),
$$

using $\left(A^{-1}\right)^{\top}\Sigma^{-1}A^{-1} = \left(A\Sigma A^{\top}\right)^{-1}$, and the normalizer picks up $\det\left(A\Sigma A^{\top}\right) = \left(\det A\right)^2\det\Sigma$. Everything matches Definition 2.6. $\blacksquare$

**Immediate corollary (sampling and whitening).** With $\Sigma = LL^{\top}$ the Cholesky factorization, $\boldsymbol\mu + L\mathbf{Z}$ has covariance $LL^{\top} = \Sigma$ — the standard sampler. Running it backwards, $L^{-1}(\mathbf{X}-\boldsymbol\mu) \sim \mathcal{N}(\mathbf{0}, I)$ is the **whitening transform**.

### Proof 3.2: Marginals of a Gaussian Are Gaussian

**Claim.** If $\mathbf{X} = (\mathbf{X}_1, \mathbf{X}_2)$ is jointly Gaussian, then $\mathbf{X}_1 \sim \mathcal{N}\left(\boldsymbol\mu_1, \Sigma_{11}\right)$.

**Proof (one line, via Theorem 2.8.1).** $\mathbf{X}_1 = \begin{pmatrix}I & 0\end{pmatrix}\mathbf{X}$ is an affine map of a Gaussian, hence Gaussian, with mean $\begin{pmatrix}I & 0\end{pmatrix}\boldsymbol\mu = \boldsymbol\mu_1$ and covariance

$$
\begin{pmatrix}I & 0\end{pmatrix}\begin{pmatrix}\Sigma_{11} & \Sigma_{12}\\ \Sigma_{21} & \Sigma_{22}\end{pmatrix}\begin{pmatrix}I\\ 0\end{pmatrix} = \Sigma_{11}. \qquad \blacksquare
$$

**Proof (by direct integration, bivariate case, to see why it works).** With standardized variables and correlation $\rho$,

$$
f(x,y) = \frac{1}{2\pi\sqrt{1-\rho^2}}\exp\left(-\frac{x^2 - 2\rho xy + y^2}{2(1-\rho^2)}\right).
$$

Complete the square in $y$: $x^2 - 2\rho xy + y^2 = (y - \rho x)^2 + x^2(1-\rho^2)$. Therefore

$$
f(x,y) = \underbrace{\frac{1}{\sqrt{2\pi}}e^{-x^2/2}}_{f_X(x)}\cdot\underbrace{\frac{1}{\sqrt{2\pi(1-\rho^2)}}\exp\left(-\frac{(y-\rho x)^2}{2(1-\rho^2)}\right)}_{f_{Y\mid X}(y \mid x)},
$$

and integrating over $y$ leaves $f_X(x) = \mathcal{N}(0,1)$ because the second factor is a normalized density in $y$. $\blacksquare$

**Bonus.** The factorization just performed *is* the conditioning theorem in the bivariate case: it shows directly that

$$
Y \mid X = x \sim \mathcal{N}\left(\rho x,\ 1-\rho^2\right),
$$

a mean linear in $x$ and a variance free of $x$ — the two hallmarks of Gaussian conditioning, and the reason linear regression is the *exact* conditional mean under joint normality rather than an approximation.

### Proof 3.3: Gaussian Conditioning via the Schur Complement

**Claim.** $\mathbf{X}_1 \mid \mathbf{X}_2 = \mathbf{x}_2 \sim \mathcal{N}\left(\boldsymbol\mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(\mathbf{x}_2-\boldsymbol\mu_2),\ \Sigma_{11}-\Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}\right)$.

**Proof (decorrelation trick).** Assume $\boldsymbol\mu = \mathbf{0}$ (subtract it otherwise). Define

$$
\mathbf{W} = \mathbf{X}_1 - \Sigma_{12}\Sigma_{22}^{-1}\mathbf{X}_2.
$$

*Step 1 — $\mathbf{W}$ is Gaussian and uncorrelated with $\mathbf{X}_2$.* It is an affine map of $\mathbf{X}$, hence Gaussian (Theorem 2.8.1), and

$$
\mathrm{Cov}\left(\mathbf{W}, \mathbf{X}_2\right) = \mathrm{Cov}\left(\mathbf{X}_1,\mathbf{X}_2\right) - \Sigma_{12}\Sigma_{22}^{-1}\mathrm{Cov}\left(\mathbf{X}_2,\mathbf{X}_2\right) = \Sigma_{12} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{22} = 0.
$$

The coefficient $\Sigma_{12}\Sigma_{22}^{-1}$ was *chosen* to make this vanish — that is the entire idea.

*Step 2 — uncorrelated Gaussians are independent.* $(\mathbf{W}, \mathbf{X}_2)$ is jointly Gaussian (both are affine functions of the same Gaussian), so Theorem 2.8.4 upgrades zero correlation to independence. Hence conditioning on $\mathbf{X}_2$ does not change the law of $\mathbf{W}$.

*Step 3 — read off the conditional law.* Since $\mathbf{X}_1 = \mathbf{W} + \Sigma_{12}\Sigma_{22}^{-1}\mathbf{X}_2$ and $\mathbf{X}_2 = \mathbf{x}_2$ is now a constant,

$$
E\left[\mathbf{X}_1 \mid \mathbf{X}_2 = \mathbf{x}_2\right] = E[\mathbf{W}] + \Sigma_{12}\Sigma_{22}^{-1}\mathbf{x}_2 = \Sigma_{12}\Sigma_{22}^{-1}\mathbf{x}_2,
$$

$$
\mathrm{Cov}\left(\mathbf{X}_1 \mid \mathbf{X}_2 = \mathbf{x}_2\right) = \mathrm{Cov}(\mathbf{W}) = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21},
$$

where the last covariance expands as $\mathrm{Cov}(\mathbf{W}) = \Sigma_{11} - 2\Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21} + \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{22}\Sigma_{22}^{-1}\Sigma_{21}$, and the last two terms combine. Restoring the means gives the stated formula. $\blacksquare$

**Three readings of the same formula.**

- **Regression**: $\Sigma_{12}\Sigma_{22}^{-1}$ is the matrix of regression coefficients of $\mathbf{X}_1$ on $\mathbf{X}_2$ — least squares is Gaussian conditioning.
- **Kalman filter**: with $\mathbf{X}_2$ a noisy observation, $K = \Sigma_{12}\Sigma_{22}^{-1}$ is precisely the Kalman gain, and $\Sigma_{1\mid2}$ is the updated error covariance.
- **Gaussian process regression**: with $\Sigma$ built from a kernel, the same two lines are the GP posterior mean and variance.

**The variance does not depend on $\mathbf{x}_2$.** Where the measurement lands changes the predicted mean but not the posterior uncertainty — an exclusively Gaussian phenomenon, and the reason experimental designs can be optimized before any data is collected.

### Proof 3.4: Uncorrelated Gaussians Are Independent — and Why Marginal Normality Is Not Enough

**Claim.** If $(\mathbf{X}_1,\mathbf{X}_2)$ is **jointly** Gaussian with $\Sigma_{12} = 0$, then $\mathbf{X}_1 \perp \mathbf{X}_2$.

**Proof.** With $\Sigma_{12} = 0$ the covariance is block diagonal, so

$$
\Sigma^{-1} = \begin{pmatrix}\Sigma_{11}^{-1} & 0\\ 0 & \Sigma_{22}^{-1}\end{pmatrix}, \qquad \det\Sigma = \det\Sigma_{11}\cdot\det\Sigma_{22}.
$$

The quadratic form therefore splits, $\left(\mathbf{x}-\boldsymbol\mu\right)^{\top}\Sigma^{-1}\left(\mathbf{x}-\boldsymbol\mu\right) = Q_1(\mathbf{x}_1) + Q_2(\mathbf{x}_2)$, and the normalizer splits multiplicatively, so

$$
f_{\mathbf{X}}(\mathbf{x}) = f_1(\mathbf{x}_1)\,f_2(\mathbf{x}_2)
$$

with each factor the corresponding marginal Gaussian density. Factorization of the joint density on a product support is exactly independence (Definition 2.3). $\blacksquare$

**Counterexample: marginally normal, jointly not.** Let $X \sim \mathcal{N}(0,1)$ and let $S$ be an independent random sign with $P(S = \pm1) = \tfrac12$; set $Y = SX$.

- *Marginals*: $Y \mid S = 1$ is $\mathcal{N}(0,1)$ and $Y \mid S = -1$ is also $\mathcal{N}(0,1)$ by symmetry, so $Y \sim \mathcal{N}(0,1)$.
- *Uncorrelated*: $E[XY] = E[S]E[X^2] = 0 \times 1 = 0$, using independence of $S$ and $X$.
- *Not independent*: $\lvert Y \rvert = \lvert X \rvert$ always, so knowing $X = 2$ forces $Y \in \{-2, 2\}$.
- *Not jointly Gaussian*: the linear combination $X + Y = X(1+S)$ equals $2X$ with probability $\tfrac12$ and $0$ with probability $\tfrac12$ — a mixture with an atom at 0, which no normal law has. This violates Definition 2.7.

**Moral.** All the strength of Theorem 2.8 comes from *joint* normality, which is a statement about every linear combination, not about each coordinate separately. Checking marginal histograms for bell shapes is not a test of joint normality.

### Proof 3.5: The Precision Matrix Encodes Conditional Independence

**Claim.** For $\mathbf{X} \sim \mathcal{N}(\mathbf{0}, \Sigma)$ with $\Lambda = \Sigma^{-1}$ and $i \ne j$: $\Lambda_{ij} = 0$ iff $X_i \perp X_j$ given all other coordinates.

**Proof.** Write the log-density in precision form:

$$
\ln f(\mathbf{x}) = -\frac{1}{2}\mathbf{x}^{\top}\Lambda\mathbf{x} + c = -\frac{1}{2}\sum_{k}\Lambda_{kk}x_k^2 - \sum_{k \lt l}\Lambda_{kl}x_kx_l + c.
$$

Fix all coordinates except $x_i$ and $x_j$; the conditional density of $(X_i, X_j)$ given the rest is proportional to the same expression viewed as a function of $(x_i, x_j)$:

$$
f\left(x_i, x_j \mid \mathbf{x}_{\setminus\{i,j\}}\right) \propto \exp\left(-\frac{1}{2}\Lambda_{ii}x_i^2 - \frac{1}{2}\Lambda_{jj}x_j^2 - \Lambda_{ij}x_ix_j + a_ix_i + a_jx_j\right),
$$

where $a_i = -\sum_{k \ne i,j}\Lambda_{ik}x_k$ collects the terms linear in $x_i$ (and similarly $a_j$). Every term factors into a function of $x_i$ times a function of $x_j$ **except** the cross term $-\Lambda_{ij}x_ix_j$. Hence the conditional density factorizes iff $\Lambda_{ij} = 0$, which by Definition 2.3 is conditional independence. $\blacksquare$

**Worked contrast.** Take a chain $X_1 \to X_2 \to X_3$ with

$$
\Lambda = \begin{pmatrix}1 & -0.5 & 0\\ -0.5 & 1.5 & -0.5\\ 0 & -0.5 & 1\end{pmatrix}.
$$

The zero at $\Lambda_{13}$ says $X_1 \perp X_3 \mid X_2$. But inverting gives a $\Sigma$ with $\Sigma_{13} \ne 0$: $X_1$ and $X_3$ *are* marginally correlated, through the shared intermediary. **Marginal and conditional independence are different questions living in different matrices** — and this is precisely why Gaussian graphical models (graphical lasso, GGM structure learning) estimate a sparse $\Lambda$ rather than a sparse $\Sigma$.

**Partial correlation.** The strength of the surviving edges is $\rho_{ij\mid\text{rest}} = -\Lambda_{ij}/\sqrt{\Lambda_{ii}\Lambda_{jj}}$, which is the correlation of the residuals after regressing both variables on all the others.

### Proof 3.6: Whitening, PCA, and the Distribution of the Mahalanobis Distance

**Claim.** For $\mathbf{X} \sim \mathcal{N}(\boldsymbol\mu, \Sigma)$ with $\Sigma \succ 0$, the squared Mahalanobis distance satisfies

$$
\Delta^2 = \left(\mathbf{X}-\boldsymbol\mu\right)^{\top}\Sigma^{-1}\left(\mathbf{X}-\boldsymbol\mu\right) \sim \chi^2_d.
$$

**Proof.** Diagonalize $\Sigma = Q\Lambda_{\text{eig}}Q^{\top}$ with $Q$ orthogonal and $\Lambda_{\text{eig}} = \mathrm{diag}(\lambda_1,\ldots,\lambda_d)$, all $\lambda_i \gt 0$. Define the whitening map

$$
\mathbf{Z} = \Lambda_{\text{eig}}^{-1/2}Q^{\top}\left(\mathbf{X}-\boldsymbol\mu\right).
$$

By affine closure, $\mathbf{Z}$ is Gaussian with mean $\mathbf{0}$ and covariance

$$
\Lambda_{\text{eig}}^{-1/2}Q^{\top}\,\Sigma\,Q\Lambda_{\text{eig}}^{-1/2} = \Lambda_{\text{eig}}^{-1/2}\Lambda_{\text{eig}}\Lambda_{\text{eig}}^{-1/2} = I,
$$

so $\mathbf{Z} \sim \mathcal{N}(\mathbf{0}, I)$: its components are i.i.d. standard normals. Substituting $\Sigma^{-1} = Q\Lambda_{\text{eig}}^{-1}Q^{\top}$,

$$
\Delta^2 = \left(\mathbf{X}-\boldsymbol\mu\right)^{\top}Q\Lambda_{\text{eig}}^{-1}Q^{\top}\left(\mathbf{X}-\boldsymbol\mu\right) = \mathbf{Z}^{\top}\mathbf{Z} = \sum_{i=1}^{d}Z_i^2 \sim \chi^2_d. \qquad \blacksquare
$$

**Geometric consequences.**

- **Confidence ellipsoids.** $\left\{\mathbf{x} : \Delta^2(\mathbf{x}) \le \chi^2_{d,1-\alpha}\right\}$ is an exact $(1-\alpha)$ region, an ellipsoid with axes along the eigenvectors of $\Sigma$ and semi-axis lengths $\sqrt{\lambda_i\,\chi^2_{d,1-\alpha}}$.
- **PCA is the same eigendecomposition.** The projection onto the top-$k$ eigenvectors of $\Sigma$ maximizes retained variance $\sum_{i\le k}\lambda_i$; whitening rescales each direction to unit variance, destroying that ordering deliberately.
- **Outlier detection.** $E\left[\Delta^2\right] = d$ and $\mathrm{Var}\left(\Delta^2\right) = 2d$, so in high dimensions typical points sit at $\Delta \approx \sqrt{d}$ — the thin-shell phenomenon again, and the reason a fixed Mahalanobis threshold must scale with $d$.
- **Cholesky as the cheap alternative.** $L^{-1}(\mathbf{X}-\boldsymbol\mu)$ with $\Sigma = LL^{\top}$ whitens just as well at $O(d^3/3)$ instead of a full eigendecomposition, and $\Delta^2 = \left\lVert L^{-1}(\mathbf{x}-\boldsymbol\mu)\right\rVert^2$ is the numerically preferred evaluation — never form $\Sigma^{-1}$ explicitly.

## 4. Computational & Algorithmic Insights

### 4.1 Never Invert a Covariance Matrix

Gaussian log-densities, GP posteriors, and Kalman updates all *look* like they need $\Sigma^{-1}$ and $\det\Sigma$. They do not. Factor once, $\Sigma = LL^{\top}$ (Cholesky, $O(d^3/3)$ and half the cost of an LU), then:

$$
\Delta^2 = \left\lVert L^{-1}(\mathbf{x}-\boldsymbol\mu)\right\rVert^2 \quad\text{(one triangular solve)}, \qquad \ln\det\Sigma = 2\sum_{i=1}^{d}\ln L_{ii}.
$$

The log-density becomes

$$
\ln f(\mathbf{x}) = -\frac{d}{2}\ln(2\pi) - \sum_i \ln L_{ii} - \frac{1}{2}\left\lVert L^{-1}(\mathbf{x}-\boldsymbol\mu)\right\rVert^2,
$$

which is both faster and far better conditioned than forming an explicit inverse. If the Cholesky fails, $\Sigma$ is not positive definite — a *useful* diagnostic that an explicit inverse would silently hide.

**Jitter.** Kernel matrices are often numerically singular; adding $\epsilon I$ with $\epsilon \sim 10^{-6}\,\mathrm{tr}(\Sigma)/d$ restores positive definiteness and is interpretable as a small observation noise.

### 4.2 Estimating and Regularizing Covariance

- **The sample covariance is bad in high dimensions.** $\hat\Sigma = \frac{1}{n-1}\sum_i(\mathbf{x}_i-\bar{\mathbf{x}})(\mathbf{x}_i-\bar{\mathbf{x}})^{\top}$ is singular whenever $n \le d$, and even for $n \approx 5d$ the Marchenko-Pastur law says its eigenvalues spread over roughly $\left(1 \pm \sqrt{d/n}\right)^2$ times the truth — top eigenvalues biased up, bottom ones down.
- **Shrinkage.** Ledoit-Wolf uses $\hat\Sigma_{\text{LW}} = (1-\alpha)\hat\Sigma + \alpha\frac{\mathrm{tr}\hat\Sigma}{d}I$ with an analytically optimal $\alpha$ — a bias-variance trade in matrix form, and the same idea as ridge regression.
- **Sparse precision.** The graphical lasso maximizes $\ln\det\Lambda - \mathrm{tr}\left(\hat\Sigma\Lambda\right) - \rho\lVert\Lambda\rVert_1$, directly estimating the conditional-independence graph of Theorem 2.9.
- **Structured covariance.** Factor models $\Sigma = BB^{\top} + D$ (with $B$ tall-thin and $D$ diagonal) reduce $O(d^2)$ parameters to $O(dk)$, and the Woodbury identity inverts them in $O(dk^2)$ — the structure behind probabilistic PCA and factor-analysis models.

### 4.3 Sampling and Conditioning at Scale

- **Sampling**: $\boldsymbol\mu + L\mathbf{z}$ with $\mathbf{z}\sim\mathcal{N}(\mathbf{0},I)$ costs $O(d^2)$ per sample after one $O(d^3)$ factorization — amortize the factorization across draws.
- **Conditioning without refactoring**: rank-one Cholesky updates handle sequentially arriving observations in $O(d^2)$, which is why online Kalman filters never redo a full decomposition.
- **Large GPs**: exact GP regression is $O(n^3)$ time and $O(n^2)$ memory. Inducing-point approximations restrict the covariance to a low-rank-plus-diagonal form, and conjugate-gradient/Lanczos methods (as in GPyTorch) compute the required solves and log-determinants using only matrix-vector products.
- **Structured Gaussians**: Kronecker structure ($\Sigma = \Sigma_1 \otimes \Sigma_2$) on grid data factorizes both the Cholesky and the log-determinant, turning $O\left((n_1n_2)^3\right)$ into $O\left(n_1^3+n_2^3\right)$.

## 5. Real-World Physics & AI/ML Applications

### 5.1 AI / Machine Learning

- **Linear regression is Gaussian conditioning.** Under joint normality of $(Y, \mathbf{X})$, $E[Y \mid \mathbf{X}] = \mu_Y + \Sigma_{Y\mathbf{X}}\Sigma_{\mathbf{XX}}^{-1}(\mathbf{X}-\boldsymbol\mu_{\mathbf{X}})$ is *exactly* linear — the OLS normal equations are Theorem 2.8.3 in disguise, and the residual variance is the Schur complement.
- **Gaussian processes.** A GP places a joint Gaussian over function values with $\Sigma_{ij} = k(x_i,x_j)$; prediction is one application of the conditioning formula, giving a posterior mean $K_{*}K^{-1}\mathbf{y}$ and variance $K_{**}-K_{*}K^{-1}K_{*}^{\top}$.
- **Kalman filtering and linear-Gaussian state-space models.** Predict (affine closure) and update (conditioning) alternate forever, and the filter stays exact precisely because the Gaussian family is closed under both operations.
- **Principal component analysis** is the eigendecomposition of $\Sigma$; probabilistic PCA and factor analysis reinterpret it as a latent-variable Gaussian model, which is what makes missing-data handling and model selection principled.
- **VAEs and diffusion models.** VAE encoders emit diagonal Gaussians and the reparameterization $\mathbf{z} = \boldsymbol\mu + \boldsymbol\sigma\odot\boldsymbol\varepsilon$ is affine closure; diffusion forward kernels are Gaussian with a closed-form $t$-step marginal $\mathcal{N}\left(\sqrt{\bar\alpha_t}\mathbf{x}_0,(1-\bar\alpha_t)I\right)$, obtained by composing affine Gaussian maps.
- **Gaussian graphical models.** Estimating a sparse precision matrix recovers conditional-independence structure in gene networks, fMRI connectivity, and portfolio risk — Theorem 2.9 turned into an algorithm.
- **Linear discriminant analysis.** Assuming class-conditional Gaussians with a *shared* $\Sigma$ makes the log-odds linear in $\mathbf{x}$; allowing different $\Sigma_k$ (QDA) makes it quadratic — the entire LDA/QDA distinction is one modeling choice about covariance sharing.

### 5.2 Physics & Engineering

- **Equilibrium fluctuations are Gaussian.** Near a stable equilibrium the energy is quadratic, $E(\mathbf{x}) \approx \frac{1}{2}\mathbf{x}^{\top}H\mathbf{x}$, so the Boltzmann distribution $e^{-\beta E}$ *is* a multivariate normal with $\Sigma = \left(\beta H\right)^{-1}$ — the Hessian of the energy is literally the precision matrix, and normal modes are its eigenvectors.
- **Sensor fusion.** Combining independent measurements with covariances $\Sigma_1, \Sigma_2$ gives a posterior precision $\Sigma_1^{-1}+\Sigma_2^{-1}$ and mean weighted by precisions — additive information, which is why precision (not variance) is the natural currency of fusion.
- **Navigation and control.** GPS/IMU fusion, SLAM, and spacecraft attitude estimation are Kalman or extended-Kalman filters, i.e. repeated Gaussian conditioning; the covariance matrix propagated alongside the estimate is the reported uncertainty ellipsoid.
- **Signal processing.** Stationary Gaussian processes have Toeplitz covariance matrices, diagonalized asymptotically by the Fourier basis (Wiener-Khinchin) — which is why spectral methods and Gaussian assumptions travel together.
- **Finance and risk.** Multivariate Gaussian return models give portfolio variance $\mathbf{w}^{\top}\Sigma\mathbf{w}$ and closed-form Value-at-Risk; their well-documented failure in crises is a *tail-dependence* failure, addressed by $t$-copulas rather than by fixing the marginals.
- **Diffusion and Langevin dynamics.** The Ornstein-Uhlenbeck process has Gaussian transition kernels with mean decaying exponentially toward 0 and a variance saturating at the stationary value — the continuous-time embodiment of affine closure.

### 5.3 Key Formula Summary

| Object | Formula | Notes |
|---|---|---|
| Joint density | $f_{X,Y}(x,y) = \partial^2F/\partial x\,\partial y$ | Determines marginals, not conversely |
| Marginal | $f_X(x) = \int f_{X,Y}(x,y)\,dy$ | Integrate out the unwanted block |
| Conditional | $f_{Y\mid X}(y \mid x) = f_{X,Y}(x,y)/f_X(x)$ | A density in $y$ for each fixed $x$ |
| Independence | $f_{X,Y} = f_Xf_Y$ on a product support | Support shape matters |
| Covariance matrix | $\Sigma = E\left[\mathbf{XX}^\top\right] - \boldsymbol\mu\boldsymbol\mu^\top \succeq 0$ | $\mathbf{a}^\top\Sigma\mathbf{a} = \mathrm{Var}(\mathbf{a}^\top\mathbf{X})$ |
| Affine map | $A\mathbf{X}+\mathbf{b} \sim \mathcal{N}\left(A\boldsymbol\mu+\mathbf{b}, A\Sigma A^\top\right)$ | Moment part needs no normality |
| MVN density | $f \propto \exp\left(-\tfrac12(\mathbf{x}-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf{x}-\boldsymbol\mu)\right)$ | Elliptical contours |
| Conditional mean | $\boldsymbol\mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(\mathbf{x}_2-\boldsymbol\mu_2)$ | Linear in the observation |
| Conditional covariance | $\Sigma_{11}-\Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$ | Schur complement; free of $\mathbf{x}_2$ |
| Precision zeros | $\Lambda_{ij} = 0 \iff X_i \perp X_j \mid \text{rest}$ | Graph structure lives in $\Sigma^{-1}$ |
| Partial correlation | $\rho_{ij\mid\text{rest}} = -\Lambda_{ij}/\sqrt{\Lambda_{ii}\Lambda_{jj}}$ | Edge weights of the GGM |
| Mahalanobis | $\Delta^2 \sim \chi^2_d$ | Exact confidence ellipsoids |
| Sampling | $\boldsymbol\mu + L\mathbf{z}$, $\Sigma = LL^\top$ | Cholesky; also gives $\ln\det\Sigma$ |
| Sklar | $F(\mathbf{x}) = C\left(F_1(x_1),\ldots,F_d(x_d)\right)$ | Marginals and dependence separate |

## 6. Canonical Literature Mapping & References

| Source | Chapters / Sections | Coverage |
|---|---|---|
| Blitzstein & Hwang, *Introduction to Probability* (2nd ed.) | Chapters 7–8 | Joint, marginal, conditional laws; multivariate transformations. |
| Casella & Berger, *Statistical Inference* (2nd ed.) | Chapter 4, Section 4.5 | Multiple random variables; the bivariate normal in detail. |
| Wasserman, *All of Statistics* | Sections 2.8–2.9, 3.3 | Compact treatment of multivariate laws and covariance. |
| Bishop, *Pattern Recognition and Machine Learning* | Section 2.3 | The Gaussian: partitioned marginals, conditionals, Bayes' theorem for linear-Gaussian models. |
| Murphy, *Probabilistic Machine Learning: An Introduction* | Chapters 3, 4.2 | Multivariate models, Gaussian inference, graphical structure. |
| Rasmussen & Williams, *Gaussian Processes for ML* | Chapter 2, Appendix A | GP regression as Gaussian conditioning; a complete identity cheat-sheet. |
| Anderson, *Multivariate Statistical Analysis* (3rd ed.) | Chapters 2–3 | Classical rigorous theory of the multivariate normal. |
| Durrett, *Probability: Theory and Examples* (5th ed.) | Section 3.9 | Multivariate normal, characteristic functions, multidimensional CLT. |
| Nelsen, *An Introduction to Copulas* (2nd ed.) | Chapters 1–2 | Sklar's theorem, dependence measures, tail dependence. |
| Horn & Johnson, *Matrix Analysis* (2nd ed.) | Sections 0.8, 7.1 | Schur complements, positive definiteness, block inversion. |

**Reading path**: Blitzstein & Hwang Ch. 7 for joint-distribution intuition, Bishop 2.3 for the four Gaussian identities in usable form, Rasmussen & Williams Appendix A as a permanent reference card, then Murphy Ch. 3 and Nelsen Ch. 1 for graphical models and for what to do when Gaussianity fails.